In [0]:
# ============================================
# NOTEBOOK: 01_bronze_ingestion
# PURPOSE: Ingest raw data and save to Bronze
# Layer using Delta Tables
# ============================================

# Create Medallion Architecture Databases
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_db")
spark.sql("CREATE DATABASE IF NOT EXISTS silver_db")
spark.sql("CREATE DATABASE IF NOT EXISTS gold_db")

print("✅ Databases created successfully!")
print("="*50)
spark.sql("SHOW DATABASES").show()

✅ Databases created successfully!
+------------------+
|      databaseName|
+------------------+
|         bronze_db|
|           default|
|           gold_db|
|information_schema|
|         silver_db|
+------------------+



In [0]:
# ============================================
# CELL 2: Download All 4 Datasets from FDIC API
# ============================================
import requests
import pandas as pd

print("Starting data download...")
print("="*50)

# ------------------------------------------------
# Dataset 1 - FDIC Financials
# Quarterly financial data of every US bank
# ------------------------------------------------
url1 = "https://banks.data.fdic.gov/api/financials?fields=REPDTE,CERT,ASSET,LNLSNET,DEP,EQTOT,NETINC,INTEXP&limit=10000&output=json"
r1 = requests.get(url1)
rows1 = [item['data'] for item in r1.json()['data']]
df_financials = spark.createDataFrame(pd.DataFrame(rows1))
print(f"✅ Dataset 1 - FDIC Financials    : {df_financials.count()} rows")

# ------------------------------------------------
# Dataset 2 - Failed Banks
# List of every bank that has failed — OUR ML LABEL
# ------------------------------------------------
url2 = "https://banks.data.fdic.gov/api/failures?fields=NAME,CERT,FAILDATE,SAVR,RESTYPE,QBFASSET,QBFCOST&limit=10000&output=json"
r2 = requests.get(url2)
rows2 = [item['data'] for item in r2.json()['data']]
df_failed = spark.createDataFrame(pd.DataFrame(rows2))
print(f"✅ Dataset 2 - Failed Banks        : {df_failed.count()} rows")

# ------------------------------------------------
# Dataset 3 - Institutions
# Bank profile, location, active/inactive status
# ------------------------------------------------
url3 = "https://banks.data.fdic.gov/api/institutions?fields=NAME,CERT,ASSET,STALP,CITY,STNAME,ACTIVE,INSTCAT,REPDTE,ESTYMD&limit=10000&output=json"
r3 = requests.get(url3)
rows3 = [item['data'] for item in r3.json()['data']]
df_institutions = spark.createDataFrame(pd.DataFrame(rows3))
print(f"✅ Dataset 3 - Institutions        : {df_institutions.count()} rows")

# ------------------------------------------------
# Dataset 4 - Summary
# Aggregate banking data — used for network building
# ------------------------------------------------
url4 = "https://banks.data.fdic.gov/api/summary?fields=REPDTE,ASSET,DEP,INTEXP,LNLSNET,NETINC,EQTOT&limit=10000&output=json"
r4 = requests.get(url4)
rows4 = [item['data'] for item in r4.json()['data']]
df_summary = spark.createDataFrame(pd.DataFrame(rows4))
print(f"✅ Dataset 4 - Summary             : {df_summary.count()} rows")

print("="*50)
print("🎉 All 4 datasets downloaded successfully!")

Starting data download...
✅ Dataset 1 - FDIC Financials    : 10000 rows
✅ Dataset 2 - Failed Banks        : 4114 rows
✅ Dataset 3 - Institutions        : 10000 rows
✅ Dataset 4 - Summary             : 7989 rows
🎉 All 4 datasets downloaded successfully!


In [0]:
# ============================================
# CELL 3: Save Raw Data as Bronze Delta Tables
# ============================================
from pyspark.sql.functions import current_timestamp, lit

print("Saving data to Bronze Layer...")
print("="*50)

# ------------------------------------------------
# Bronze Table 1 - Financials
# ------------------------------------------------
df_financials \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source", lit("FDIC_API")) \
    .withColumn("layer", lit("bronze")) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze_db.bronze_financials")
print("✅ bronze_financials saved!")

# ------------------------------------------------
# Bronze Table 2 - Failed Banks
# ------------------------------------------------
df_failed \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source", lit("FDIC_API")) \
    .withColumn("layer", lit("bronze")) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze_db.bronze_failed_banks")
print("✅ bronze_failed_banks saved!")

# ------------------------------------------------
# Bronze Table 3 - Institutions
# ------------------------------------------------
df_institutions \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source", lit("FDIC_API")) \
    .withColumn("layer", lit("bronze")) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze_db.bronze_institutions")
print("✅ bronze_institutions saved!")

# ------------------------------------------------
# Bronze Table 4 - Summary
# ------------------------------------------------
df_summary \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source", lit("FDIC_API")) \
    .withColumn("layer", lit("bronze")) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze_db.bronze_summary")
print("✅ bronze_summary saved!")

print("="*50)
print("🎉 All Bronze Tables saved successfully!")

Saving data to Bronze Layer...
✅ bronze_financials saved!
✅ bronze_failed_banks saved!
✅ bronze_institutions saved!
✅ bronze_summary saved!
🎉 All Bronze Tables saved successfully!


In [0]:
# ============================================
# CELL 4: Verify Bronze Layer
# ============================================
print("Verifying Bronze Layer...")
print("="*50)

# Show all tables in bronze_db
print("📋 Tables in Bronze Database:")
spark.sql("SHOW TABLES IN bronze_db").show()

# Check row counts
print("📊 Row Counts:")
print(f"  bronze_financials  : {spark.table('bronze_db.bronze_financials').count()} rows")
print(f"  bronze_failed_banks: {spark.table('bronze_db.bronze_failed_banks').count()} rows")
print(f"  bronze_institutions: {spark.table('bronze_db.bronze_institutions').count()} rows")
print(f"  bronze_summary     : {spark.table('bronze_db.bronze_summary').count()} rows")

# Preview bronze_financials
print("\n📊 Bronze Financials Preview:")
display(spark.table("bronze_db.bronze_financials").limit(3))

# Preview bronze_failed_banks
print("\n📊 Bronze Failed Banks Preview:")
display(spark.table("bronze_db.bronze_failed_banks").limit(3))

print("="*50)
print("🎉 Bronze Layer Verification Complete!")

Verifying Bronze Layer...
📋 Tables in Bronze Database:
+---------+-------------------+-----------+
| database|          tableName|isTemporary|
+---------+-------------------+-----------+
|bronze_db|bronze_failed_banks|      false|
|bronze_db|  bronze_financials|      false|
|bronze_db|bronze_institutions|      false|
|bronze_db|     bronze_summary|      false|
+---------+-------------------+-----------+

📊 Row Counts:
  bronze_financials  : 10000 rows
  bronze_failed_banks: 4114 rows
  bronze_institutions: 10000 rows
  bronze_summary     : 7989 rows

📊 Bronze Financials Preview:


REPDTE,ASSET,CERT,NETINC,LNLSNET,DEP,EQTOT,ID,ingestion_timestamp,source,layer
19840331,27693,10002,63,15902,24983,1796,10002_19840331,2026-03-12T18:22:25.561Z,FDIC_API,bronze
19840630,27399,10002,122,16876,25207,1838,10002_19840630,2026-03-12T18:22:25.561Z,FDIC_API,bronze
19840930,45739,10002,305,30274,42221,3102,10002_19840930,2026-03-12T18:22:25.561Z,FDIC_API,bronze



📊 Bronze Failed Banks Preview:


QBFASSET,FAILDATE,CERT,RESTYPE,SAVR,NAME,ID,ingestion_timestamp,source,layer
374.0,5/28/1934,null,FAILURE,FDIC,FON DU LAC STATE BANK,1,2026-03-12T18:22:37.892Z,FDIC_API,bronze
2305.0,1/3/1935,null,FAILURE,FDIC,CLIFFSIDE PARK TITLE GUARANTEE & TRUST CO.,10,2026-03-12T18:22:37.892Z,FDIC_API,bronze
null,12/21/1936,null,FAILURE,FDIC,THE CUMMINGS STATE BANK,100,2026-03-12T18:22:37.892Z,FDIC_API,bronze


🎉 Bronze Layer Verification Complete!
